# 实际收益率 Qlib 日历前缀修复

只修正从已验 NAS 原件映射到原 Stage1 前两个训练日的可用性；零新行情请求、零收益读取、零拟合、零账户。

In [1]:
import csv,hashlib,json,os,tempfile
from datetime import datetime
from pathlib import Path
import pandas as pd
import qlib
from qlib.data import D
from qlib.workflow import R
from src.data.nas_market_data import archive_nas_recorder,materialize_nas_evidence

root=Path.cwd();card=root/'research/stage1_treasury_real_yield_source_20261023'
repair=json.loads((card/'map_repair_plan.json').read_text());source=json.loads((card/'source_receipt.json').read_text());binding=json.loads((card/'nas_binding.json').read_text());sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert repair['frozen'] and not (card/'map_repair_state.json').exists()
assert source['plan_sha256']==repair['parent_plan_sha256'] and binding['verified'] and binding['source_id']==repair['source_NAS_id']
for obj in binding['objects']:
    assert sha(root/obj['local_path'])==obj['sha256'] and (root/obj['local_path']).stat().st_size==obj['bytes']
raw=root/'data/snapshots/stage1_treasury_real_yield_source_20261023';rows=[]
for receipt in source['request_receipts']:
    p=raw/receipt['name'];assert sha(p)==receipt['raw_sha256']
    with p.open(newline='',encoding='utf-8-sig') as handle:
        reader=csv.DictReader(handle);columns=reader.fieldnames;record=list(reader)
    dcol=next(x for x in columns if x.strip().lower()=='date');vcol=next(x for x in columns if x.strip().lower()=='10 yr')
    rows.extend((pd.Timestamp(datetime.strptime(r[dcol],'%m/%d/%Y').date()),float(r[vcol])) for r in record)
rates=pd.Series(dict(rows)).sort_index();assert len(rates)==len(rows)==5956
state={'status':'started','plan_sha256':sha(card/'map_repair_plan.json'),'new_external_GETs':0,'SOXX_outcome_reads':0,'new_model_fits':0,'new_native_accounts':0}
(card/'map_repair_state.json').write_text(json.dumps(state,indent=2)+'\n')
os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
provider=root/'data/qlib/full_path_tiered_allocation_20261013'
experiment=repair['phase'].replace('one_mechanical_Qlib_calendar_prefix_repair_no_refetch','stage1_treasury_real_yield_source_20261023_map_repair')
qlib.init(provider_uri=str(provider),region='us',exp_manager={'class':'MLflowExpManager','module_path':'qlib.workflow.expm','kwargs':{'uri':str(card/'qlib_recorder'),'default_exp_name':experiment}})
cal=pd.DatetimeIndex(D.calendar(start_time='2018-10-01',end_time='2026-10-07',freq='day'))
records=[]
for i,day in enumerate(cal):
    bound=cal[i-2] if i>=2 else pd.NaT
    j=rates.index.searchsorted(bound,side='right')-1 if pd.notna(bound) else -1
    records.append({'qlib_session':day,'source_date':str(rates.index[j].date()) if j>=0 else None,'ten_year_real_yield_percent_lag2':float(rates.iloc[j]) if j>=0 else float('nan')})
all_map=pd.DataFrame(records)
train=all_map[(all_map.qlib_session>=pd.Timestamp('2018-10-15'))&(all_map.qlib_session<=pd.Timestamp('2022-12-30'))]
held=all_map[(all_map.qlib_session>=pd.Timestamp('2023-01-03'))&(all_map.qlib_session<=pd.Timestamp('2026-10-07'))]
assert len(train)==1061 and len(held)==944 and train.ten_year_real_yield_percent_lag2.notna().all() and held.ten_year_real_yield_percent_lag2.notna().all()
assert str(train.qlib_session.iloc[0].date())=='2018-10-15' and str(held.qlib_session.iloc[-1].date())=='2026-10-07'
map_frame=pd.concat([train,held],ignore_index=True)
coverage={'train_sessions':len(train),'train_known':int(train.ten_year_real_yield_percent_lag2.notna().sum()),'held_sessions':len(held),'held_known':int(held.ten_year_real_yield_percent_lag2.notna().sum()),'first_two_source_dates':train.source_date.iloc[:2].tolist(),'first_public_and_revision_qualified':False,'source_rows':len(rates)}
with R.start(experiment_name=experiment,uri=str(card/'qlib_recorder')):
    rec=R.get_recorder();rec.save_objects(artifact_path='inputs',repair_plan=repair);rec.save_objects(artifact_path='analysis',source_map=map_frame,coverage=coverage);rec.log_metrics(train_known=coverage['train_known'],held_known=coverage['held_known'],new_model_fits=0,new_native_accounts=0);rid=rec.id;eid=str(rec.experiment_id)
copy_binding=card/'nas_binding_map_repair.json';copy_binding.write_text(json.dumps(binding,indent=2)+'\n')
evidence=archive_nas_recorder(copy_binding,workspace_root=root,card=card,recorder_id=rid,experiment_id=eid)
with tempfile.TemporaryDirectory(prefix='treasury-map-repair-cold-') as temp:
    materialize_nas_evidence(copy_binding,card=Path(temp))
    relative=Path('qlib_recorder')/eid/rid/'artifacts/analysis/source_map'
    assert sha(card/relative)==sha(Path(temp)/relative)
result={'decision':'corrected_Stage1_10y_real_yield_source_map_full_coverage_PIT_unqualified','parent_recorder_id':repair['parent_recorder_id'],'recorder_id':rid,'experiment_id':eid,'coverage':coverage,'NAS_recorder_sha256':evidence['sha256'],'new_external_GETs':0,'SOXX_outcome_reads':0,'new_model_fits':0,'new_native_accounts':0,'qualified_forward_days':0}
(card/'map_repair_result.json').write_text(json.dumps(result,indent=2)+'\n')
state.update(status='completed',recorder_id=rid);(card/'map_repair_state.json').write_text(json.dumps(state,indent=2)+'\n')
print(json.dumps(result))


AssertionError: 

第一次日历修复未通过：配套 Qlib provider 最早为 2018-10-12，2018-10-15 只有一个可核证的前序交易日，计划要求全 1061 日有值过严。未产生新 Recorder；见 map_repair_attempt1_failure.json。
